# E008 — Drift, staleness and exposure re-balancing

**Hypotheses:** H9 ([docs/hypotheses.md](../../docs/hypotheses.md))

**Question:** What drifts across the ten days, what does a stale model cost, and can an adaptation layer spread exposure without losing CTR?

**Method:** Daily PSI against the training window, churn statistics, a staleness backtest of models frozen at different days, a causal per-genre hourly recalibration, and a replay of a cohort-exposure penalty with doubly robust CTR estimates.

In [1]:
from charade.analysis.experiment import setup

ctx = setup()
f"mode: {'smoke (fixture, tiny model)' if ctx.smoke else 'full data'}"

'mode: full data'

In [2]:
import matplotlib.pyplot as plt
from IPython.display import Markdown

from charade.analysis import adaptation, drift
from charade.analysis.experiment import ensure_bundle

ensure_bundle(ctx)
tables = drift.run(ctx.settings, ctx.data_dir, ctx.reports / "drift")
tables["Daily mix and churn"]

day,impressions,ctr,first_seen_character_share,top100_jaccard_prev_day,character_hhi,new_creative_share
str,i64,f64,f64,f64,f64,f64
"""2014-10-21""",111065,0.179183,1.0,NaN,0.000656,1.0
"""2014-10-22""",143836,0.166627,0.010846,0.754386,0.000618,0.34433
"""2014-10-23""",104452,0.191638,0.005256,0.652893,0.000602,0.22282
"""2014-10-24""",89378,0.184329,0.005012,0.538462,0.000586,0.132214
"""2014-10-25""",90218,0.192545,0.008402,0.481481,0.000585,0.025749
"""2014-10-26""",103948,0.194972,0.005281,0.538462,0.000583,0.026369
"""2014-10-27""",86788,0.195465,0.003365,0.5625,0.000594,0.133786
"""2014-10-28""",142909,0.165357,0.007081,0.550388,0.000608,0.342407
"""2014-10-29""",104450,0.172705,0.003973,0.680672,0.000629,0.235807


In [3]:
stale = tables["Staleness: frozen models scored on later days (1 seed each)"]
fig, ax = plt.subplots(figsize=(6, 4))
for (day,), part in stale.group_by("scored_day", maintain_order=True):
    ax.plot(part["age_days"], part["ne"], "o-", label=str(day))
ax.set(xlabel="model age (days since last training day)", ylabel="NE", title="Staleness")
ax.legend(title="scored day", fontsize=7)
plt.show()

/tmp/ipykernel_2481804/550794735.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
adaptation.run(ctx.settings, ctx.data_dir, ctx.reports / "drift" / "adaptation.md")
Markdown((ctx.reports / "drift" / "adaptation.md").read_text())

# Adaptation layer (generated by `uv run poe adapt`)

Cohort-exposure penalty vs greedy pCTR (lambda = 0); half-life 2000 cohort impressions; DR with the independent reward model.

Selection rule (pre-registered, validation day only): largest lambda with estimated change >= -0.2 pp and an interval containing 0. **Chosen lambda = 8.**

## Validation day (selection)

| lambda | dr_ctr | dr_delta_vs_greedy | ci_low | ci_high | cohort_dominance | cohort_hhi | repeat_exposure |
|---|---|---|---|---|---|---|---|
| 0.0000 | 0.1853 | 0.0000 | 0.0000 | 0.0000 | 0.0948 | 0.0349 | 0.1001 |
| 1.0000 | 0.1940 | 0.0086 | 0.0024 | 0.0147 | 0.0667 | 0.0283 | 0.0973 |
| 2.0000 | 0.1943 | 0.0090 | 0.0022 | 0.0150 | 0.0604 | 0.0257 | 0.0956 |
| 4.0000 | 0.1902 | 0.0049 | -0.0039 | 0.0139 | 0.0559 | 0.0230 | 0.0931 |
| 8.0000 | 0.1852 | -0.0002 | -0.0092 | 0.0082 | 0.0537 | 0.0203 | 0.0909 |
| 16.0000 | 0.1820 | -0.0034 | -0.0144 | 0.0080 | 0.0545 | 0.0177 | 0.0869 |

## Test days (confirmation, 102,874 replayed impressions)

| lambda | dr_ctr | dr_delta_vs_greedy | ci_low | ci_high | cohort_dominance | cohort_hhi | repeat_exposure |
|---|---|---|---|---|---|---|---|
| 0.0000 | 0.1839 | 0.0000 | 0.0000 | 0.0000 | 0.0768 | 0.0271 | 0.1047 |
| 1.0000 | 0.1857 | 0.0017 | -0.0028 | 0.0065 | 0.0644 | 0.0231 | 0.1021 |
| 2.0000 | 0.1830 | -0.0010 | -0.0079 | 0.0056 | 0.0593 | 0.0213 | 0.0998 |
| 4.0000 | 0.1859 | 0.0020 | -0.0065 | 0.0098 | 0.0549 | 0.0193 | 0.0973 |
| 8.0000 | 0.1866 | 0.0027 | -0.0059 | 0.0108 | 0.0478 | 0.0167 | 0.0937 |
| 16.0000 | 0.1841 | 0.0001 | -0.0093 | 0.0095 | 0.0412 | 0.0144 | 0.0914 |


## Reading
- The CTR level moves for every genre together; character preferences are stable.
- NE worsens with model age on every scored day, hence daily retraining.
- lambda is chosen on the validation day by the pre-registered rule; the test days only confirm it.
- The replay uses logged exposure history, so it can show "no CTR loss" but never a fatigue benefit.

## Conclusion
Ads rotate fast (13-37 % new creatives a day) and a frozen model loses ~0.003 NE per day, so retrain daily; online recalibration slightly worsens NE on the validation day; the exposure penalty selected on the validation day by a pre-registered rule (lambda = 8) cuts cohort HHI 38 % on test with CTR change +0.27 pp [-0.59, +1.08].

**Decision:** Daily retraining with gated promotion; no online recalibration; exposure penalty at lambda = 8 (chosen on validation) pending an online A/B test.